In [2]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

# Locate the project root
project_root = Path.cwd().parents[2]

# Add project root to Python's module search path
sys.path.insert(0, str(project_root))

# Import the reusable data loader
from src.data_loader import load_cv_data

In [3]:
fold_datasets = load_cv_data(project_root)

for fold_number, (train_df, val_df) in enumerate(fold_datasets, start=1):
    print(
        f"Fold {fold_number}: "
        f"Training = {train_df.shape}, "
        f"Validation = {val_df.shape}"
    )

Fold 1: Training = (177324, 7), Validation = (42659, 7)
Fold 2: Training = (176273, 7), Validation = (43710, 7)
Fold 3: Training = (170186, 7), Validation = (49797, 7)
Fold 4: Training = (178311, 7), Validation = (41672, 7)
Fold 5: Training = (177838, 7), Validation = (42145, 7)


In [4]:
# Inspect the skill information
fold_train_df, fold_val_df = fold_datasets[0]

print("Columns:")
print(fold_train_df.columns.tolist())

print("\nSkill ID examples:")
print(fold_train_df["skill_id"].head(10).tolist())

print("\nSkill set examples:")
print(fold_train_df["skill_set"].head(10).tolist())

print("\nSkill set data type:")
print(type(fold_train_df["skill_set"].iloc[0]))

Columns:
['user_id', 'order_id', 'problem_id', 'correct', 'skill_id', 'skill_set', 'attempt_count']

Skill ID examples:
['2_37_70', '2_37_70', '2_37_70', '2_37_70', '2_37_70', '2_37_70', '2_37_48_77', '2_48_79', '2_37_48', '2_48_79']

Skill set examples:
["{'70', '2', '37'}", "{'70', '2', '37'}", "{'70', '2', '37'}", "{'70', '2', '37'}", "{'70', '2', '37'}", "{'70', '2', '37'}", "{'2', '48', '37', '77'}", "{'2', '48', '79'}", "{'2', '48', '37'}", "{'2', '48', '79'}"]

Skill set data type:
<class 'str'>


In [5]:
def convert_skill_id(skill_id):
    if pd.isna(skill_id):
        return set()

    return set(str(skill_id).split("_"))


for train_df, val_df in fold_datasets:
    train_df["skill_set"] = train_df["skill_id"].apply(convert_skill_id)
    val_df["skill_set"] = val_df["skill_id"].apply(convert_skill_id)

In [6]:
fold_train_df, fold_val_df = fold_datasets[0]

print("Original skill IDs:")
print(fold_train_df["skill_id"].head(5).tolist())

print("\nConverted skill sets:")
print(fold_train_df["skill_set"].head(5).tolist())

print("\nData type:")
print(type(fold_train_df["skill_set"].iloc[0]))

print("\nMissing skill IDs:")
print(fold_train_df["skill_id"].isna().sum())

Original skill IDs:
['2_37_70', '2_37_70', '2_37_70', '2_37_70', '2_37_70']

Converted skill sets:
[{'2', '70', '37'}, {'2', '70', '37'}, {'2', '70', '37'}, {'2', '70', '37'}, {'2', '70', '37'}]

Data type:
<class 'set'>

Missing skill IDs:
11163


In [7]:

import pandas as pd
import numpy as np


def add_same_skill_history_features(data, p0, alpha=10, training=False):
    """
    Calculate binary and fractional same-skill historical features.

    Binary overlap:
        A previous question contributes 1 if it shares at least
        one skill with the current question.

    Fractional overlap:
        A previous question contributes:
        |previous skills ∩ current skills| / |previous skills|

    Only earlier interactions from the same student are used.
    """

    df = data.copy()
    df = df.sort_values(["user_id", "order_id"]).copy()

    # Convert skill_id into actual Python sets.
    # Missing skill IDs become empty sets.
    skill_sets = df["skill_id"].apply(
        lambda x: set(str(x).split("_")) if pd.notna(x) else set()
    ).tolist()

    # Prepare output arrays
    binary_experience = []
    binary_correct = []

    fractional_experience = []
    fractional_correct = []

    # Leakage-safe prior
    if training:
        total_correct = df["correct"].sum()
        total_interactions = len(df)

        row_prior = (
            total_correct - df["correct"].to_numpy()
        ) / (total_interactions - 1)
    else:
        row_prior = np.full(len(df), p0)

    # Map sorted dataframe positions to their prior values
    prior_values = iter(row_prior)

    # Process each student independently
    for _, student_df in df.groupby("user_id", sort=False):

        previous_history = []

        for idx in student_df.index:

            current_skills = skill_sets[df.index.get_loc(idx)]

            binary_n = 0
            binary_c = 0.0

            fractional_n = 0.0
            fractional_c = 0.0

            # Use only previous questions
            for previous_skills, previous_outcome in previous_history:

                if not current_skills or not previous_skills:
                    continue

                overlap = current_skills & previous_skills

                if overlap:
                    # V2.1 / V2.3: Binary overlap
                    binary_n += 1
                    binary_c += previous_outcome

                    # V2.2 / V2.4: Fractional overlap
                    weight = len(overlap) / len(previous_skills)

                    fractional_n += weight
                    fractional_c += weight * previous_outcome

            binary_experience.append(binary_n)
            binary_correct.append(binary_c)

            fractional_experience.append(fractional_n)
            fractional_correct.append(fractional_c)

            # Add the current question only AFTER calculating
            # its features, preventing current-target leakage.
            current_outcome = df.loc[idx, "correct"]
            previous_history.append(
                (current_skills, current_outcome)
            )

    # Attach features in the same sorted row order
    df["same_skill_previous_interactions"] = binary_experience
    df["same_skill_previous_correct"] = binary_correct

    df["weighted_same_skill_previous_interactions"] = fractional_experience
    df["weighted_same_skill_previous_correct"] = fractional_correct

    # Convert prior values into an aligned Series
    priors = pd.Series(row_prior, index=df.index)

    # Smoothed binary same-skill accuracy
    df["same_skill_smoothed_accuracy"] = (
        df["same_skill_previous_correct"] + alpha * priors
    ) / (
        df["same_skill_previous_interactions"] + alpha
    )

    # Smoothed fractional same-skill accuracy
    df["weighted_same_skill_smoothed_accuracy"] = (
        df["weighted_same_skill_previous_correct"] + alpha * priors
    ) / (
        df["weighted_same_skill_previous_interactions"] + alpha
    )

    # Log-transformed experience features
    df["log_same_skill_previous_interactions"] = np.log1p(
        df["same_skill_previous_interactions"]
    )

    df["log_weighted_same_skill_previous_interactions"] = np.log1p(
        df["weighted_same_skill_previous_interactions"]
    )

    return df
